In [1]:
import os
import gc
import cmath
import random
import math
import json
from pathlib import Path

import pandas as pd

from PIL import Image, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt
from scipy import ndimage
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split




In [ ]:
# ============================================================
# 1. ENVIRONMENT + PACKAGE CHECK
# ============================================================

import os, gc, math, time, random, copy, warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from PIL import Image

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Check PyTorch BEFORE installing/using PennyLane
# ------------------------------------------------------------
print("PyTorch:", torch.__version__)
print("PyTorch path:", torch.__file__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("torch._utils available:", hasattr(torch, "_utils"))

assert hasattr(torch, "_utils"), (
    "PyTorch installation is broken. Restart the Kaggle session "
    "before continuing. Do NOT reinstall torch/torchvision."
)

# ------------------------------------------------------------
# Install PennyLane
# ------------------------------------------------------------
!pip install -q pennylane

import pennylane as qml

print("PennyLane:", qml.__version__)

# ------------------------------------------------------------
# Test torchvision + DenseNet BEFORE running the full pipeline
# ------------------------------------------------------------
print("\nTesting DenseNet121...")

test_model = models.densenet121(weights="DEFAULT")

print("DenseNet121 loaded successfully.")

del test_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# Download BreakHis
# ------------------------------------------------------------
import kagglehub

path = kagglehub.dataset_download("ambarish/breakhis")

print("\nPath to dataset files:", path)

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("waseemalastal/breakhis-breast-cancer-histopathological-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/waseemalastal/breakhis-breast-cancer-histopathological-dataset


In [ ]:
import os, re, gc, copy, time, math, random, warnings
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef,
)

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------- CONFIG -----------------------------
CFG = dict(
    DATASET_PATH="/kaggle/input/datasets/waseemalastal/breakhis-breast-cancer-histopathological-dataset/BreakHis - Breast Cancer Histopathological Database/dataset_cancer_v1/dataset_cancer_v1/classificacao_binaria",
    MAG="400X",
    SEED=42,
    N_FOLDS=5,
    VAL_FRAC=0.20,
    START_FOLD=0,             # 0-indexed. Set to 3 to resume at fold 4 after folds 1-3 finished
                              # (previous results are reloaded from the saved CSVs).
    EPOCHS=35,
    PATIENCE=9,
    MIN_EPOCHS=8,
    BATCH_SIZE=16,            # T4-friendly; increase to 32 if memory allows (must be >= 2)
    LR=7e-4,                  # head/reducer/fusion LR
    BACKBONE_LR=1e-5,         # fine-tuned DenseNet block LR
    QLR=2e-3,                 # quantum circuit parameter LR
    WD=1e-4,
    N_QUBITS=8,               # exact statevector; 16 qubits is 256x larger state
    Q_LAYERS=2,
    FUSION_Q_LAYERS=2,
    UNFREEZE_FROM="denseblock4",
    NUM_WORKERS=2,
    OUT_DIR="/kaggle/working",
    USE_PRETRAINED=True,
    USE_AMP=True,
    USE_EMA=True,
    EMA_DECAY=0.995,
    LABEL_SMOOTH=0.05,
    GRAD_CLIP=1.0,
    THRESHOLD_MIN=0.20,
    THRESHOLD_MAX=0.80,
    THRESHOLD_STEP=0.025,
    PIN_MEMORY=True,
    RUN_EFFICIENCY_REPORT=False,  # expensive; enable after training if required
)
BATCH_SIZE = CFG["BATCH_SIZE"]
IM_MEAN = [0.485, 0.456, 0.406]
IM_STD = [0.229, 0.224, 0.225]
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
KINDS = ["classical", "classical_ens", "vqc_a", "vqc_b", "concat_q", "vqefm"]
QNAMES = ("qa", "qb", "qf")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Stable experiments where possible; benchmark speed may be slightly lower.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["SEED"])
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Config:", CFG)

# ----------------------- 1. FIND DATASET --------------------------
SEARCH_ROOTS = [Path(CFG["DATASET_PATH"]), Path("/kaggle/input"), Path("/content"), Path(".")]
dataset_candidates = []
for root in SEARCH_ROOTS:
    if root.exists():
        if root.is_dir() and root.name.lower() == "classificacao_binaria":
            dataset_candidates.append(root.resolve())
        else:
            try:
                dataset_candidates.extend(p.resolve() for p in root.rglob("classificacao_binaria") if p.is_dir())
            except (PermissionError, OSError):
                pass
# unique and retain candidates with expected magnification directories
unique_candidates = {str(p): p for p in dataset_candidates}
valid_roots = [p for p in unique_candidates.values() if any((p / m).is_dir() for m in ("40X", "100X", "200X", "400X"))]
if not valid_roots:
    raise FileNotFoundError("Could not find classificacao_binaria containing 40X/100X/200X/400X. Attach BreakHis to the notebook and check CFG['DATASET_PATH'].")
DATASET_ROOT = valid_roots[0]
print("Dataset root:", DATASET_ROOT)

# ----------------------- 2. IMAGE INDEX ----------------------------
records = []
for mag_dir in DATASET_ROOT.iterdir():
    if not mag_dir.is_dir() or mag_dir.name.upper() not in {"40X", "100X", "200X", "400X"}:
        continue
    mag = mag_dir.name.upper()
    for class_dir in mag_dir.iterdir():
        if not class_dir.is_dir():
            continue
        diagnosis = class_dir.name.strip().lower()
        if diagnosis not in {"benign", "malignant"}:
            continue
        label = 0 if diagnosis == "benign" else 1
        for img_path in class_dir.rglob("*"):
            if img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({"image_path": str(img_path), "diagnosis": diagnosis,
                                "label": label, "magnification": mag})
df_all = pd.DataFrame(records).drop_duplicates(subset=["image_path"]).reset_index(drop=True)
if df_all.empty:
    raise ValueError(f"No supported image files found below {DATASET_ROOT}")
print("All images:", len(df_all))
print("Images by magnification:\n", df_all["magnification"].value_counts().sort_index())

selected_mag = CFG["MAG"].upper()
if selected_mag not in {"40X", "100X", "200X", "400X"}:
    raise ValueError("CFG['MAG'] must be 40X, 100X, 200X, or 400X")
df = df_all[df_all["magnification"] == selected_mag].copy().reset_index(drop=True)
if df.empty:
    raise ValueError(f"No images for magnification {selected_mag}")

# BreakHis names usually look like SOB_B_A-14-22549AB-100-001.png.
def extract_patient_id(image_path):
    stem = Path(image_path).stem
    match = re.match(r"^(SOB_[BM]_[A-Z]+-\d+-\d+[A-Z]*)-(\d+)-(\d+)$", stem, flags=re.IGNORECASE)
    if match:
        return match.group(1).upper()
    # Fail loudly rather than risk patient-level leakage with a wrong grouping.
    raise ValueError(f"Unexpected BreakHis filename: {Path(image_path).name}. Update extract_patient_id() to match the actual filename format.")

df["patient_id"] = df["image_path"].apply(extract_patient_id)
label_counts = df.groupby("patient_id")["label"].nunique()
if (label_counts > 1).any():
    raise ValueError(f"Conflicting labels for specimen IDs: {label_counts[label_counts > 1].index.tolist()[:10]}")
patient_df = df.groupby("patient_id", as_index=False).agg(label=("label", "first"))
print(f"Selected MAG={selected_mag} | images={len(df)} | patients/specimens={len(patient_df)}")
print("Image labels:\n", df["diagnosis"].value_counts())
print("Patient labels:\n", patient_df["label"].value_counts().sort_index())
if patient_df["label"].nunique() != 2 or patient_df["label"].value_counts().min() < CFG["N_FOLDS"]:
    raise ValueError("Both classes need at least N_FOLDS patient/specimen groups for stratified CV.")

# ----------------------- 3. PATIENT-LEVEL FOLDS --------------------
def make_folds(patients, n_folds, val_frac, seed):
    # Standard stratified 5-fold CV: each fold uses 80% of patients for training
    # and the remaining 20% as validation. There is no separate test split.
    if n_folds != 5:
        print(f"Using {n_folds}-fold CV; each validation fold is approximately {100/n_folds:.1f}%.")
    pids = patients["patient_id"].to_numpy()
    labels = patients["label"].to_numpy()
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for tr_idx, val_idx in skf.split(pids, labels):
        folds.append({"train": set(pids[tr_idx]), "val": set(pids[val_idx])})
    return folds

folds = make_folds(patient_df, CFG["N_FOLDS"], CFG["VAL_FRAC"], CFG["SEED"])
for k, fold in enumerate(folds):
    assert not (fold["train"] & fold["val"])
    assert fold["train"] | fold["val"] == set(patient_df["patient_id"])
    print(f"Fold {k+1}: train patients={len(fold['train'])}, val patients={len(fold['val'])}")
assert set().union(*(f["val"] for f in folds)) == set(patient_df["patient_id"])
print("Patient/specimen-level 5-fold train/validation checks passed.")

# ----------------------- 4. TRANSFORMS / DATASET ------------------
def stain_norm_safe(img):
    # Identity RGB fallback, matching the supplied notebook. Add a validated
    # stain-normalization algorithm here only if it is applied consistently.
    if not isinstance(img, Image.Image):
        img = Image.fromarray(np.asarray(img))
    return img.convert("RGB")

train_tf = transforms.Compose([
    transforms.Lambda(stain_norm_safe),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(IM_MEAN, IM_STD),
])
eval_tf = transforms.Compose([
    transforms.Lambda(stain_norm_safe),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IM_MEAN, IM_STD),
])

class BreakHisDS(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths = list(paths)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.tf = tf
        if len(self.paths) != len(self.labels):
            raise ValueError("Image paths and labels have different lengths")
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = im.convert("RGB")
        if self.tf is not None:
            img = self.tf(img)
        return img, int(self.labels[i])

paths = df["image_path"].to_numpy()
y = df["label"].to_numpy(dtype=np.int64)
pid_arr = df["patient_id"].to_numpy()

def make_fold_loaders(fold, cfg):
    # BatchNorm1d in the reducer cannot train on a batch of size 1.
    assert cfg["BATCH_SIZE"] >= 2, "BatchNorm needs batch size >= 2 in training"
    train_df = df[df.patient_id.isin(fold["train"])].copy()
    val_df = df[df.patient_id.isin(fold["val"])].copy()
    assert set(train_df.patient_id).isdisjoint(val_df.patient_id)
    assert set(train_df.patient_id) | set(val_df.patient_id) == set(patient_df.patient_id)
    tr_ds = BreakHisDS(train_df.image_path, train_df.label, train_tf)
    va_ds = BreakHisDS(val_df.image_path, val_df.label, eval_tf)
    # Class-balanced sampling is applied ONLY to training images.
    counts = np.bincount(train_df.label.to_numpy(), minlength=2).astype(np.float64)
    class_weights = 1.0 / np.maximum(counts, 1.0)
    sample_weights = class_weights[train_df.label.to_numpy()]
    sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double),
                                    num_samples=len(sample_weights), replacement=True)
    common = dict(batch_size=cfg["BATCH_SIZE"], num_workers=cfg["NUM_WORKERS"],
                  pin_memory=(cfg["PIN_MEMORY"] and torch.cuda.is_available()),
                  persistent_workers=(cfg["NUM_WORKERS"] > 0))
    # FIX: drop_last=True on the TRAIN loader avoids a final batch of size 1
    # (e.g. 1457 % 16 == 1), which crashes BatchNorm1d in training mode.
    # Safe here because the sampler draws with replacement, so a different
    # random tail is dropped every epoch.
    tr_loader = DataLoader(tr_ds, sampler=sampler, shuffle=False, drop_last=True, **common)
    # Validation runs in eval() mode, so a batch of 1 is fine and every image is scored.
    va_loader = DataLoader(va_ds, shuffle=False, drop_last=False, **common)
    return tr_loader, va_loader, train_df, val_df

# ----------------------- 5. MULTI-SCALE DENSENET ------------------
class MultiScaleDenseNet201(nn.Module):
    """DenseNet201 with pooled descriptors from all four dense blocks."""
    def __init__(self, pretrained=True, unfreeze_from="denseblock4", proj_dim=128):
        super().__init__()
        try:
            weights = models.DenseNet201_Weights.DEFAULT if pretrained else None
            net = models.densenet201(weights=weights)
        except Exception as e:
            print("Pretrained weight loading unavailable; using model defaults:", repr(e))
            net = models.densenet201(weights=None)
        self.features = net.features
        self.proj1 = nn.Sequential(nn.Conv2d(256, proj_dim, 1, bias=False), nn.BatchNorm2d(proj_dim), nn.ReLU(inplace=True))
        self.proj2 = nn.Sequential(nn.Conv2d(512, proj_dim, 1, bias=False), nn.BatchNorm2d(proj_dim), nn.ReLU(inplace=True))
        self.proj3 = nn.Sequential(nn.Conv2d(1792, proj_dim, 1, bias=False), nn.BatchNorm2d(proj_dim), nn.ReLU(inplace=True))
        self.proj4 = nn.Sequential(nn.Conv2d(1920, proj_dim, 1, bias=False), nn.BatchNorm2d(proj_dim), nn.ReLU(inplace=True))
        self.level_score = nn.Sequential(nn.Linear(proj_dim, 32), nn.Tanh(), nn.Linear(32, 1))
        self.aggregate = nn.Sequential(
            nn.Linear(4 * proj_dim, 1024), nn.LayerNorm(1024), nn.ReLU(inplace=True), nn.Dropout(0.10)
        )
        self._configure_trainable(unfreeze_from)

    def _configure_trainable(self, unfreeze_from):
        for p in self.features.parameters():
            p.requires_grad = False
        # Explicit module selection avoids fragile lexical comparisons.
        module_names = ["denseblock4", "norm5"]
        if unfreeze_from == "denseblock3":
            module_names = ["denseblock3", "transition3", "denseblock4", "norm5"]
        elif unfreeze_from == "denseblock2":
            module_names = ["denseblock2", "transition2", "denseblock3", "transition3", "denseblock4", "norm5"]
        for name, module in self.features.named_children():
            if name in module_names:
                for p in module.parameters():
                    p.requires_grad = True
        # Projections, gates and aggregate are newly initialized and trainable.
        for module in [self.proj1, self.proj2, self.proj3, self.proj4, self.level_score, self.aggregate]:
            for p in module.parameters():
                p.requires_grad = True

    def forward(self, x):
        f = self.features
        x = f.conv0(x); x = f.norm0(x); x = f.relu0(x); x = f.pool0(x)
        x1 = f.denseblock1(x)
        x = f.transition1(x1)
        x2 = f.denseblock2(x)
        x = f.transition2(x2)
        x3 = f.denseblock3(x)
        x = f.transition3(x3)
        x4 = f.denseblock4(x)
        x4 = f.norm5(x4)
        x4 = F.relu(x4, inplace=False)
        descs = []
        for feat, proj in zip((x1, x2, x3, x4), (self.proj1, self.proj2, self.proj3, self.proj4)):
            descs.append(F.adaptive_avg_pool2d(proj(feat), 1).flatten(1))
        # Soft attention over levels; concatenated descriptors retain level-specific detail.
        stack = torch.stack(descs, dim=1)                # [B,4,proj_dim]
        weights = torch.softmax(self.level_score(stack).squeeze(-1), dim=1).unsqueeze(-1)
        weighted = stack * (0.5 + weights)               # gentle gating, not destructive averaging
        return self.aggregate(weighted.flatten(1))       # [B,1024]


def make_reducer(bottleneck, p=0.30):
    return nn.Sequential(
        nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(p),
        nn.Linear(512, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(p),
        nn.Linear(128, bottleneck), nn.Tanh()
    )

def make_head(bottleneck, p=0.30):
    return nn.Sequential(
        nn.Linear(bottleneck, bottleneck), nn.LayerNorm(bottleneck), nn.GELU(),
        nn.Dropout(p), nn.Linear(bottleneck, 2)
    )

# ----------------------- 6. EFFICIENT BATCHED QUANTUM LAYERS -------
class TorchStatevectorCircuit(nn.Module):
    """Batched, differentiable exact statevector circuit implemented in PyTorch.

    It eliminates Python/PennyLane QNode calls per sample. Complexity remains
    exponential in qubit count, hence the practical 8-qubit default.
    """
    def __init__(self, n_qubits=8, n_layers=2, topology="ring"):
        super().__init__()
        if n_qubits < 2:
            raise ValueError("n_qubits must be >= 2")
        self.n_qubits = int(n_qubits)
        self.n_layers = int(n_layers)
        self.dim = 1 << self.n_qubits
        self.topology = topology
        self.ry_weights = nn.Parameter(0.05 * torch.randn(n_layers, n_qubits))
        self.rz_weights = nn.Parameter(0.05 * torch.randn(n_layers, n_qubits))
        idx = torch.arange(self.dim, dtype=torch.long)
        self.register_buffer("basis_index", idx, persistent=False)
        z_signs = []
        for q in range(n_qubits):
            bit = (idx >> q) & 1
            z_signs.append((1.0 - 2.0 * bit.float()))
        self.register_buffer("z_signs", torch.stack(z_signs, dim=1), persistent=False)  # [dim,nq]
        perms = []
        for layer in range(n_layers):
            if topology == "ring":
                pairs = [(q, (q + 1) % n_qubits) for q in range(n_qubits)]
            elif topology == "staggered":
                offset = 1 if layer % 2 == 0 else 2
                pairs = [(q, (q + offset) % n_qubits) for q in range(n_qubits)]
            else:
                raise ValueError("topology must be ring or staggered")
            layer_perms = []
            for control, target in pairs:
                # CNOT is self-inverse; index-select by transformed basis index.
                target_mask = 1 << target
                perm = torch.where(((idx >> control) & 1).bool(), idx ^ target_mask, idx)
                layer_perms.append(perm)
            perms.append(layer_perms)
        # Register each permutation as a non-persistent buffer (device follows module).
        self._perm_names = []
        for li, layer_perms in enumerate(perms):
            names = []
            for pi, perm in enumerate(layer_perms):
                name = f"_perm_{li}_{pi}"
                self.register_buffer(name, perm, persistent=False)
                names.append(name)
            self._perm_names.append(names)

    def _single_ry(self, state, theta, q):
        b = state.shape[0]
        shaped = state.reshape((b,) + (2,) * self.n_qubits)
        axis = 1 + (self.n_qubits - 1 - q)  # q=0 is least-significant basis bit
        perm_axes = [0, axis] + [a for a in range(1, self.n_qubits + 1) if a != axis]
        inv_axes = [perm_axes.index(a) for a in range(len(perm_axes))]
        pair = shaped.permute(perm_axes).contiguous().reshape(b, 2, -1)
        c = torch.cos(theta * 0.5).to(state.real.dtype).unsqueeze(1)
        s = torch.sin(theta * 0.5).to(state.real.dtype).unsqueeze(1)
        a0, a1 = pair[:, 0], pair[:, 1]
        out = torch.stack((c * a0 - s * a1, s * a0 + c * a1), dim=1)
        out = out.reshape((b, 2) + (2,) * (self.n_qubits - 1)).permute(inv_axes).contiguous()
        return out.reshape(b, self.dim)

    def _single_rz(self, state, theta, q):
        b = state.shape[0]
        shaped = state.reshape((b,) + (2,) * self.n_qubits)
        axis = 1 + (self.n_qubits - 1 - q)
        perm_axes = [0, axis] + [a for a in range(1, self.n_qubits + 1) if a != axis]
        inv_axes = [perm_axes.index(a) for a in range(len(perm_axes))]
        pair = shaped.permute(perm_axes).contiguous().reshape(b, 2, -1)
        half = theta * 0.5
        p0 = torch.complex(torch.cos(half), -torch.sin(half)).unsqueeze(1)
        p1 = torch.complex(torch.cos(half), torch.sin(half)).unsqueeze(1)
        out = torch.stack((p0 * pair[:, 0], p1 * pair[:, 1]), dim=1)
        out = out.reshape((b, 2) + (2,) * (self.n_qubits - 1)).permute(inv_axes).contiguous()
        return out.reshape(b, self.dim)

    def forward(self, angles):
        # Keep the simulator in float32 even under mixed-precision CNN training.
        # Complex rotations are not reliably supported in float16 autocast.
        angles = angles.float()
        # angles [B,n_qubits], clipped to keep encoding well-conditioned.
        angles = torch.pi * torch.tanh(angles)
        b = angles.shape[0]
        state = torch.zeros((b, self.dim), dtype=torch.complex64, device=angles.device)
        state[:, 0] = 1.0 + 0.0j
        for q in range(self.n_qubits):
            state = self._single_ry(state, angles[:, q], q)
        for li in range(self.n_layers):
            for q in range(self.n_qubits):
                state = self._single_ry(state, self.ry_weights[li, q].expand(b), q)
                state = self._single_rz(state, self.rz_weights[li, q].expand(b), q)
            for name in self._perm_names[li]:
                state = state.index_select(1, getattr(self, name))
        probs = state.real.square() + state.imag.square()
        # Force measurement reduction to float32 even inside CNN AMP autocast.
        with torch.autocast(device_type=angles.device.type, enabled=False):
            return probs.float() @ self.z_signs.float()

class QLayerA(nn.Module):
    def __init__(self):
        super().__init__()
        self.circuit = TorchStatevectorCircuit(CFG["N_QUBITS"], CFG["Q_LAYERS"], "ring")
    def forward(self, x):
        return self.circuit(x)

class QLayerB(nn.Module):
    def __init__(self):
        super().__init__()
        self.circuit = TorchStatevectorCircuit(CFG["N_QUBITS"], CFG["Q_LAYERS"], "staggered")
    def forward(self, x):
        return self.circuit(x)

class QLayerF(nn.Module):
    """Lightweight variational quantum-inspired fusion of measured observables.

    It reuses the two circuit measurement vectors instead of simulating a third
    full statevector circuit, reducing quantum-variant cost substantially.
    """
    def __init__(self):
        super().__init__()
        n = CFG["N_QUBITS"]
        self.gate = nn.Sequential(nn.Linear(3 * n, 2 * n), nn.GELU(), nn.Linear(2 * n, n), nn.Sigmoid())
        self.mix = nn.Sequential(nn.Linear(3 * n, n), nn.Tanh(), nn.LayerNorm(n))
    def forward(self, a, b):
        joined = torch.cat([a, b, a * b], dim=-1)
        g = self.gate(joined)
        mixed = self.mix(joined)
        return torch.tanh(g * a + (1.0 - g) * b + 0.25 * mixed)

# ----------------------- 7. HYBRID MODEL VARIANTS ------------------
class Hybrid(nn.Module):
    def __init__(self, kind, bottleneck=None):
        super().__init__()
        if kind not in KINDS:
            raise ValueError(f"Unknown model kind {kind}; choose from {KINDS}")
        self.kind = kind
        self.bottleneck = int(bottleneck or CFG["N_QUBITS"])
        self.backbone = MultiScaleDenseNet201(
            pretrained=CFG["USE_PRETRAINED"], unfreeze_from=CFG["UNFREEZE_FROM"]
        )
        self.reducer = make_reducer(self.bottleneck)
        n = self.bottleneck
        if kind == "vqc_a":
            self.qa = QLayerA()
        elif kind == "vqc_b":
            self.qb = QLayerB()
        elif kind in ("concat_q", "vqefm"):
            self.qa, self.qb = QLayerA(), QLayerB()
            if kind == "vqefm":
                self.qf = QLayerF()
            else:
                self.fuse = nn.Sequential(nn.Linear(2 * n, n), nn.LayerNorm(n), nn.Tanh())
        elif kind == "classical_ens":
            self.b1 = nn.Sequential(nn.Linear(n, n), nn.Tanh(), nn.LayerNorm(n))
            self.b2 = nn.Sequential(nn.Linear(n, n), nn.GELU(), nn.LayerNorm(n))
            self.fuse = nn.Sequential(nn.Linear(2 * n, n), nn.LayerNorm(n), nn.Tanh())
        self.clf = make_head(n)

    def forward(self, imgs):
        features = self.backbone(imgs)
        z = self.reducer(features)
        k = self.kind
        if k == "classical":
            fused = z
        elif k == "classical_ens":
            fused = self.fuse(torch.cat([self.b1(z), self.b2(z)], dim=-1))
        else:
            if z.shape[-1] != CFG["N_QUBITS"]:
                raise RuntimeError("Reducer output dimension must match N_QUBITS for quantum variants")
            if k == "vqc_a":
                fused = self.qa(z)
            elif k == "vqc_b":
                fused = self.qb(z)
            else:
                qa, qb = self.qa(z), self.qb(z)
                fused = self.qf(qa, qb) if k == "vqefm" else self.fuse(torch.cat([qa, qb], dim=-1))
        return self.clf(fused)


def n_params(model):
    return sum(p.numel() for p in model.parameters())
def n_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
def n_qparams(model):
    return sum(p.numel() for name, p in model.named_parameters() if name.split(".")[0] in QNAMES)

# ----------------------- 8. EMA / OPTIMIZER ------------------------
class ModelEMA:
    def __init__(self, model, decay=0.995):
        self.ema = copy.deepcopy(model).eval()
        self.decay = float(decay)
        for p in self.ema.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            src = msd[k].detach()
            if v.is_floating_point():
                v.mul_(self.decay).add_(src, alpha=1.0 - self.decay)
            else:
                v.copy_(src)

def make_optimizer(model):
    backbone_params, quantum_params, other_params = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone.features."):
            backbone_params.append(p)
        elif name.split(".")[0] in QNAMES:
            quantum_params.append(p)
        else:
            other_params.append(p)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": CFG["BACKBONE_LR"], "weight_decay": CFG["WD"]})
    if other_params:
        groups.append({"params": other_params, "lr": CFG["LR"], "weight_decay": CFG["WD"]})
    if quantum_params:
        groups.append({"params": quantum_params, "lr": CFG["QLR"], "weight_decay": CFG["WD"]})
    return torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)

# ----------------------- 9. METRICS / THRESHOLD --------------------
def metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / max(tn + fp, 1)
    try:
        auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else float("nan")
    except ValueError:
        auc = float("nan")
    return {
        "acc": accuracy_score(y_true, pred),
        "bacc": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, pred, zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, pred) if len(np.unique(y_true)) > 1 else 0.0,
        "threshold": float(threshold),
    }

def optimize_threshold(y_true, prob):
    # Tune only on validation probabilities; never optimize on test data.
    candidates = np.arange(CFG["THRESHOLD_MIN"], CFG["THRESHOLD_MAX"] + 1e-9, CFG["THRESHOLD_STEP"])
    best_t, best_score = 0.5, -float("inf")
    for t in candidates:
        m = metrics(y_true, prob, float(t))
        # balanced accuracy first; F1 breaks ties / favors useful positive detection
        score = 0.70 * m["bacc"] + 0.30 * m["f1"]
        if score > best_score:
            best_t, best_score = float(t), float(score)
    return best_t, best_score

@torch.no_grad()
def predict_proba(model, loader, device, use_amp=None):
    model.eval()
    probs, labels = [], []
    amp_on = CFG["USE_AMP"] and device.type == "cuda" if use_amp is None else use_amp
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_on):
            logits = model(xb)
        probs.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy().tolist())
        labels.extend(yb.numpy().tolist())
    return np.asarray(probs), np.asarray(labels, dtype=int)

def validation_score(y_true, probs):
    m = metrics(y_true, probs, 0.5)
    auc = m["auc"] if np.isfinite(m["auc"]) else m["bacc"]
    return 0.55 * auc + 0.45 * m["bacc"]

# ----------------------- 10. TRAINING ------------------------------
def train_model(kind, train_loader, val_loader, seed, device):
    seed_everything(seed)
    model = Hybrid(kind).to(device)
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["EPOCHS"], eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["LABEL_SMOOTH"])
    amp_on = CFG["USE_AMP"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_on)
    ema = ModelEMA(model, CFG["EMA_DECAY"]) if CFG["USE_EMA"] else None
    best_score, best_epoch, best_state = -float("inf"), 0, None
    best_threshold = 0.5
    bad_epochs = 0
    for epoch in range(1, CFG["EPOCHS"] + 1):
        model.train()
        running_loss, n_seen = 0.0, 0
        t0 = time.time()
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_on):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            if CFG["GRAD_CLIP"] is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
            scaler.step(optimizer)
            scaler.update()
            if ema is not None:
                ema.update(model)
            running_loss += float(loss.detach()) * len(yb)
            n_seen += len(yb)
        scheduler.step()
        eval_model = ema.ema if ema is not None else model
        val_prob, val_y = predict_proba(eval_model, val_loader, device)
        score = validation_score(val_y, val_prob)
        threshold, threshold_score = optimize_threshold(val_y, val_prob)
        vm = metrics(val_y, val_prob, threshold)
        print(f"  {kind:12s} epoch {epoch:02d}/{CFG['EPOCHS']} loss={running_loss/max(n_seen,1):.4f} "
              f"val_acc={vm['acc']*100:.2f}% val_bAcc={vm['bacc']*100:.2f}% val_F1={vm['f1']*100:.2f}% "
              f"val_AUC={vm['auc']:.4f} thr={threshold:.3f} score={score:.4f} time={time.time()-t0:.1f}s")
        if score > best_score + 1e-5:
            best_score, best_epoch, best_threshold = score, epoch, threshold
            source = eval_model.state_dict()
            best_state = {k: v.detach().cpu().clone() for k, v in source.items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if epoch >= CFG["MIN_EPOCHS"] and bad_epochs >= CFG["PATIENCE"]:
            print(f"  Early stopping at epoch {epoch}; best epoch={best_epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(device).eval()
    model._best_epoch = best_epoch
    model._best_threshold = best_threshold
    model._best_val_score = best_score
    # Keep EMA-selected checkpoint weights in the returned model.
    print(f"  Best {kind}: epoch={best_epoch}, val_score={best_score:.4f}, threshold={best_threshold:.3f}")
    return model, best_epoch

# ----------------------- 11. DEVICE / GRAD CHECK ------------------
def run_device_check(model, xb, yb, device):
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
    model = model.to(device)
    bad = [(n, str(p.device)) for n, p in model.named_parameters() if p.device.type != device.type]
    if bad: raise AssertionError(f"Parameters on wrong device: {bad[:10]}")
    xb, yb = xb.to(device), yb.to(device)
    was_training = model.training
    model.train(); model.zero_grad(set_to_none=True)
    out = model(xb)
    loss = F.cross_entropy(out, yb)
    loss.backward()
    missing = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
    print("logits:", tuple(out.shape), "loss:", float(loss.detach()), "trainable parameters without grad:", len(missing))
    if missing: print("Examples without gradients:", missing[:15])
    model.zero_grad(set_to_none=True); model.train(was_training)

# ----------------------- 12. EXPERIMENT LOOP -----------------------
METRICS_CSV = os.path.join(CFG["OUT_DIR"], "cv_fold_metrics_v2.csv")
OOF_CSV = os.path.join(CFG["OUT_DIR"], "cv_oof_predictions_v2.csv")

rows, oof_parts = [], []
START_FOLD = int(CFG["START_FOLD"])
if START_FOLD > 0:
    # Resume: reload fully completed folds (< START_FOLD) from the saved CSVs.
    # Any partial results from the crashed fold are dropped by the filter.
    if not (os.path.exists(METRICS_CSV) and os.path.exists(OOF_CSV)):
        raise FileNotFoundError("START_FOLD > 0 but previous result CSVs were not found in OUT_DIR.")
    res_prev = pd.read_csv(METRICS_CSV)
    oof_prev = pd.read_csv(OOF_CSV)
    rows = res_prev[res_prev["fold"] < START_FOLD].to_dict("records")
    oof_parts = [oof_prev[oof_prev["fold"] < START_FOLD]]
    print(f"Resuming from fold {START_FOLD + 1}: reloaded {len(rows)} metric rows "
          f"and {len(oof_parts[0])} OOF predictions from folds 1-{START_FOLD}.")

for fold_idx, fold in enumerate(folds):
    if fold_idx < START_FOLD:
        continue
    train_loader, val_loader, train_df, val_df = make_fold_loaders(fold, CFG)
    print(f"\n{'='*80}\nFOLD {fold_idx+1}/{CFG['N_FOLDS']} | train={len(train_df)} val={len(val_df)} images\n{'='*80}")
    for kind in KINDS:
        start = time.time()
        model, best_epoch = train_model(kind, train_loader, val_loader,
                                        seed=CFG["SEED"] + fold_idx, device=device)
        # The held-out 20% validation fold supplies this fold's CV predictions.
        val_prob, val_y = predict_proba(model, val_loader, device)
        threshold = float(getattr(model, "_best_threshold", 0.5))
        m_img = metrics(val_y, val_prob, threshold)
        val_meta = val_df[["patient_id", "label"]].reset_index(drop=True)
        patient_pred = pd.DataFrame({"pid": val_meta.patient_id, "y": val_y, "p": val_prob})
        patient_pred = patient_pred.groupby("pid", as_index=False).agg(y=("y", "first"), p=("p", "mean"))
        patient_threshold, _ = optimize_threshold(patient_pred.y.to_numpy(), patient_pred.p.to_numpy())
        m_pat = metrics(patient_pred.y.to_numpy(), patient_pred.p.to_numpy(), patient_threshold)
        rows.append(dict(model=kind, fold=fold_idx, level="image", best_epoch=best_epoch, **m_img))
        rows.append(dict(model=kind, fold=fold_idx, level="patient", best_epoch=best_epoch, **m_pat))
        tmp = pd.DataFrame({"model": kind, "fold": fold_idx, "pid": val_meta.patient_id,
                            "y": val_y, "p": val_prob, "threshold": threshold,
                            "patient_threshold": patient_threshold})
        oof_parts.append(tmp)
        print(f"RESULT {kind:12s} | best_ep={best_epoch:02d} | val img acc={m_img['acc']*100:5.2f}% "
              f"bAcc={m_img['bacc']*100:5.2f}% AUC={m_img['auc']:.4f} | val patient acc={m_pat['acc']*100:5.2f}% "
              f"patient AUC={m_pat['auc']:.4f} | {time.time()-start:.0f}s")
        # Fold checkpoint: contains the trained model and validation-selected threshold.
        ckpt_path = os.path.join(CFG["OUT_DIR"], f"{kind}_fold{fold_idx+1}_best.pth")
        torch.save({"model_kind": kind, "state_dict": model.state_dict(), "config": CFG,
                    "best_epoch": best_epoch, "image_threshold": threshold,
                    "patient_threshold": patient_threshold, "magnification": selected_mag}, ckpt_path)
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    # Written only after ALL models of the fold finish, so the CSVs never hold a partial fold.
    pd.DataFrame(rows).to_csv(METRICS_CSV, index=False)
    pd.concat(oof_parts, ignore_index=True).to_csv(OOF_CSV, index=False)

res = pd.DataFrame(rows)
oof = pd.concat(oof_parts, ignore_index=True)
res.to_csv(METRICS_CSV, index=False)
oof.to_csv(OOF_CSV, index=False)

# ----------------------- 13. SUMMARY TABLES ------------------------
metric_cols = ["acc", "bacc", "precision", "recall", "specificity", "f1", "auc", "mcc"]
summary = (res.groupby(["model", "level"])[metric_cols]
           .agg(["mean", "std"]).round(4))
print("\nCross-validation summary (mean ± std across folds):")
print(summary)
summary.to_csv(os.path.join(CFG["OUT_DIR"], "cv_summary_mean_std.csv"))

# OOF patient-level scores: aggregate each specimen's held-out validation-fold predictions.
patient_oof = (oof.groupby(["model", "pid"], as_index=False)
               .agg(y=("y", "first"), p=("p", "mean"), threshold=("threshold", "mean"),
                    patient_threshold=("patient_threshold", "mean")))
patient_rows = []
for kind, sub in patient_oof.groupby("model"):
    thr = float(sub.patient_threshold.median())
    patient_rows.append({"model": kind, **metrics(sub.y.to_numpy(), sub.p.to_numpy(), thr)})
patient_oof_metrics = pd.DataFrame(patient_rows).sort_values("auc", ascending=False)
print("\nOOF patient-level summary:")
print(patient_oof_metrics.round(4).to_string(index=False))
patient_oof_metrics.to_csv(os.path.join(CFG["OUT_DIR"], "oof_patient_metrics.csv"), index=False)

# ----------------------- 14. OPTIONAL EFFICIENCY REPORT ------------
def model_size_mb(m):
    nbytes = sum(p.numel() * p.element_size() for p in m.parameters())
    nbytes += sum(b.numel() * b.element_size() for b in m.buffers())
    return nbytes / (1024 ** 2)

@torch.no_grad()
def measure_latency(m, n_warmup=3, n_runs=10):
    m = m.to(device).eval()
    x = torch.randn(1, 3, 224, 224, device=device)
    for _ in range(n_warmup): m(x)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_runs): m(x)
    if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_runs * 1000

if CFG["RUN_EFFICIENCY_REPORT"]:
    efficiency_rows = []
    for kind in KINDS:
        print("Efficiency profile:", kind)
        m = Hybrid(kind).to(device).eval()
        try:
            latency = measure_latency(m)
            gpu_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if device.type == "cuda" else 0.0
            efficiency_rows.append({"model": kind, "params": n_params(m), "trainable_params": n_trainable_params(m),
                                    "quantum_params": n_qparams(m), "model_size_mb": model_size_mb(m),
                                    "latency_ms_batch1": latency, "peak_gpu_mb": gpu_mb,
                                    "note": "Custom statevector ops are not fully counted by standard THOP FLOP profilers."})
        finally:
            del m; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    pd.DataFrame(efficiency_rows).to_csv(os.path.join(CFG["OUT_DIR"], "efficiency_metrics.csv"), index=False)
    print(pd.DataFrame(efficiency_rows).to_string(index=False))

print("\nFinished. Files saved in:", CFG["OUT_DIR"])
print(" - cv_fold_metrics_v2.csv")
print(" - cv_oof_predictions_v2.csv")
print(" - cv_summary_mean_std.csv")
print(" - oof_patient_metrics.csv")
print(" - per-fold best checkpoints")

: 